In [1]:
from docx import Document

In [4]:
doc = Document("../tests/docx/test_1.docx")

In [9]:
from docx import Document
from docx.oxml.ns import qn


def traverse_paragraph_with_media(paragraph):
    """
    Traverses a paragraph and extracts text, images, and formulas by inspecting XML.
    Preserves the order of elements as they appear in the document.
    """
    content = []

    # Iterate through all child elements of the paragraph's XML element
    for child in paragraph._element.iterchildren():
        # 1. Handle Text Runs (<w:r>) and Drawing/Math within them
        if child.tag == qn("w:r"):
            # A. Check for Images (Drawings) inside the run
            # Images are typically inside <w:drawing> tags within a run
            drawings = child.findall(qn("w:drawing"))
            for drawing in drawings:
                try:
                    # Find the blip element which contains the relationship ID (r:embed)
                    # Namespace for drawingml
                    ns_a = {"a": "http://schemas.openxmlformats.org/drawingml/2006/main"}
                    ns_r = {"r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships"}

                    blip = drawing.xpath(".//a:blip/@r:embed", namespaces={**ns_a, **ns_r})
                    if blip:
                        rId = blip[0]
                        # Get the image part from the document's related parts
                        image_part = paragraph.part.related_parts[rId]
                        image_bytes = image_part.blob
                        # Extract extension from content type (e.g., 'image/png' -> 'png')
                        content_type = image_part.content_type
                        image_ext = content_type.split("/")[-1] if "/" in content_type else "bin"

                        content.append({"type": "image", "data": image_bytes, "ext": image_ext})
                except Exception as e:
                    print(f"Warning: Could not extract image from run: {e}")

            # B. Check for Formulas (<m:oMath>) inside the run
            oMath = child.find(qn("m:oMath"))
            if oMath is not None:
                content.append({"type": "formula", "data": oMath.xml})

            # C. Extract Text from the run
            # itertext() yields strings directly, so we just join them
            text_parts = [node for node in child.itertext()]
            text = "".join(text_parts)

            if text:
                content.append({"type": "text", "data": text})

        # 2. Handle Formulas that might be direct children of the paragraph (less common)
        elif child.tag == qn("m:oMath"):
            content.append({"type": "formula", "data": child.xml})

        # 3. Handle other elements like tables (<w:tbl>) if they appear as siblings
        # (Note: Tables are usually separate from paragraphs in docx structure,
        # but sometimes custom XML or specific structures might vary)

    return content


# Usage
doc = Document("../tests/docx/test_1.docx")

for paragraph in doc.paragraphs:
    items = traverse_paragraph_with_media(paragraph)
    for item in items:
        if item["type"] == "text":
            print(f"Text: {item['data']}")
        elif item["type"] == "image":
            print(f"Image found: {len(item['data'])} bytes, format: {item['ext']}")
            # Example: Save image
            # with open(f"output_image.{item['ext']}", 'wb') as f:
            #     f.write(item['data'])
        elif item["type"] == "formula":
            # Print first 100 chars of formula XML to avoid clutter
            print(f"Formula XML: {item['data'][:100]}...")

Text: 什么是学习？什么是学习？
Text: I compress, therefore I am.I compress, therefore I am.
Text: --Anonymous--Anonymous
Text: ++
Text: 摘要摘要
Text: 所罗门诺夫在1964年发表了被后人称为所罗门诺夫归纳法（所罗门诺夫在1964年发表了被后人称为所罗门诺夫归纳法（
Text: SolomonoffSolomonoff
Text:  Induction）的重要工作。所罗门诺夫归纳法可以被视为广义学习机制。它可以被当作大语言模型的理论基础，可以解释GPT的核心机制next token prediction。最后，本文还讨论了SKC理论在人工智能领域中可能的应用以及哲学涵义。 Induction）的重要工作。所罗门诺夫归纳法可以被视为广义学习机制。它可以被当作大语言模型的理论基础，可以解释GPT的核心机制next token prediction。最后，本文还讨论了SKC理论在人工智能领域中可能的应用以及哲学涵义。
Text: 背景背景
Text: 我们由此总结出一种新的三个世界模型如下如所示。我们由此总结出一种新的三个世界模型如下如所示。
Text: 800100182880ObserveObserveObserveCompressCompressCompressExperimentExperimentExperiment00ObserveObserveObserveCompressCompressCompressExperimentExperimentExperiment
Text: 2127250709930Real WorldReal WorldReal World00Real WorldReal WorldReal World
Text: 9842502557780DataDDataata00DataDDataata
Text: 35433002569210ModelMModelodel0ModelMModelodel
Text: 美国最原创的哲学家皮尔士（美国最原创的哲学家皮尔士（
Text: Charles Sanders PeirceCharles Sanders Peirce
Text: ，，
Text: 1839-19141839-19

AttributeError: 'lxml.etree._Element' object has no attribute 'xml'